# 08 Detection To FoodLens Pipeline

This notebook connects multi-food detection outputs to the existing FoodLens classifier. The detector proposes candidate food regions; each crop is classified with the ResNet50 FT-V2 Food-101 model and then passed through the decision-layer logic.

This is the bridge notebook before adding multi-food outputs to the app.

## 1. Imports And Setup

All imports are centralized in this cell. The notebook expects crop outputs from Notebook 7 and FoodLens model artifacts from Kaggle inputs.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
import json
from pathlib import Path
from typing import Any

import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F
from torch import nn
from torchvision import models, transforms

## 2. Configuration

Set the detection output path and FoodLens artifact path here. Kaggle model and notebook-output paths may need to be adjusted after attaching inputs.

In [ ]:
@dataclass(frozen=True)
class Config:
    SEED: int = 42
    DETECTION_DIR: Path = Path("/kaggle/working/results/multi_food_detection")
    ARTIFACT_DIR: Path = Path(
        "/kaggle/input/models/tuannm3823/food101-baseline-artifacts/"
        "pytorch/default/1/food101-baseline-artifacts"
    )
    MODEL_PATH: Path = ARTIFACT_DIR / "resnet50_ft_v2_best.pth"
    CLASS_NAMES_PATH: Path = ARTIFACT_DIR / "class_names.json"
    CALIBRATION_PATH: Path = ARTIFACT_DIR / "calibration.json"
    OUTPUT_DIR: Path = Path(
        "/kaggle/working/results/multi_food_foodlens_pipeline"
    )
    IMAGE_SIZE: int = 224
    TOP_K: int = 5
    AUTO_CONFIDENCE: float = 0.70
    SUGGEST_CONFIDENCE: float = 0.35


CFG = Config()
torch.manual_seed(CFG.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG.SEED)
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CFG

## 3. Load FoodLens Classifier

The classifier is the existing ResNet50 FT-V2 Food-101 model. Detection adds localization; it does not replace the classifier in this notebook.

In [ ]:
def read_json(path: Path, default: Any) -> Any:
    """Read JSON from disk with a fallback value.

    Args:
        path: JSON file path.
        default: Value returned when the file does not exist.

    Returns:
        Parsed JSON content or the fallback value.
    """
    if not path.exists():
        return default
    return json.loads(path.read_text())


def make_classifier_head(in_features: int) -> nn.Sequential:
    """Create the project-standard Food-101 classifier head.

    Args:
        in_features: Input feature size from the backbone.

    Returns:
        A three-layer classifier head for 101 Food-101 classes.
    """
    return nn.Sequential(
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256, 101),
    )


def load_classifier() -> tuple[nn.Module, list[str], float, torch.device]:
    """Load the FoodLens classifier and metadata.

    Returns:
        Model, class names, calibrated temperature, and execution device.
    """
    class_names = read_json(CFG.CLASS_NAMES_PATH, [])
    if len(class_names) != 101:
        raise ValueError("class_names.json must contain 101 Food-101 labels.")

    calibration = read_json(CFG.CALIBRATION_PATH, {})
    temperature = float(calibration.get("temperature", 1.0))
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = models.resnet50(weights=None)
    model.fc = make_classifier_head(model.fc.in_features)
    model.load_state_dict(torch.load(CFG.MODEL_PATH, map_location=device))
    model.to(device)
    model.eval()
    return model, class_names, temperature, device


model, class_names, temperature, device = load_classifier()
temperature, device

## 4. Classify Detection Crops

Each crop receives top-k Food-101 predictions. The decision band is intentionally simple here; later app integration can reuse the full decision-policy artifacts.

In [ ]:
preprocess = transforms.Compose(
    [
        transforms.Resize((CFG.IMAGE_SIZE, CFG.IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)


def classify_crop(crop_path: Path) -> list[tuple[str, float]]:
    """Predict top-k Food-101 classes for one detected crop.

    Args:
        crop_path: Path to an exported crop image.

    Returns:
        Ordered `(class_name, confidence)` predictions.
    """
    image = Image.open(crop_path).convert("RGB")
    image_tensor = preprocess(image).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(image_tensor).cpu()
        probabilities = F.softmax(logits / temperature, dim=1)
        top_probabilities, top_indices = probabilities.topk(CFG.TOP_K, dim=1)
    return [
        (class_names[class_index], float(confidence))
        for class_index, confidence in zip(
            top_indices[0].tolist(),
            top_probabilities[0].tolist(),
        )
    ]


def decision_band(top_confidence: float) -> str:
    """Map top-1 confidence to a simple crop-level decision band."""
    if top_confidence >= CFG.AUTO_CONFIDENCE:
        return "auto_accept"
    if top_confidence >= CFG.SUGGEST_CONFIDENCE:
        return "suggest"
    return "confirm"

## 5. Export Multi-Food Predictions

The final table joins detector metadata with FoodLens crop predictions. This table is the target shape for later backend and frontend integration.

In [ ]:
detections_path = CFG.DETECTION_DIR / "detections.csv"
detections_df = pd.read_csv(detections_path)

prediction_rows: list[dict[str, object]] = []
for row in detections_df.itertuples(index=False):
    predictions = classify_crop(Path(row.crop_path))
    top_label, top_confidence = predictions[0]
    output = row._asdict()
    output.update(
        {
            "foodlens_top_label": top_label,
            "foodlens_top_confidence": top_confidence,
            "decision_band": decision_band(top_confidence),
            "top_k_predictions": json.dumps(predictions),
        }
    )
    prediction_rows.append(output)

multi_food_df = pd.DataFrame(prediction_rows)
multi_food_df.to_csv(CFG.OUTPUT_DIR / "multi_food_predictions.csv", index=False)
multi_food_df.head()

## 6. App Integration Notes

The app should consume `multi_food_predictions.csv` or a matching API response shape later. Each detected region needs a bounding box, crop path, top-k predictions, confidence, and decision band.